# Batch encoding

- The output of a tokenizer isn’t a simple Python dictionary; what we get is actually a special `BatchEncoding` object.
- It’s a **subclass of a dictionary** (which is why we were able to index into that result without any problem before), but with additional methods that are mostly used by fast tokenizers.

- Besides their **parallelization** capabilities, the key functionality of fast tokenizers is that they always **keep track of the original span** of texts the final tokens come from — a feature we call __offset mapping__.
    - This in turn unlocks features like
        - mapping each word to the tokens it generated
        - or mapping each character of the original text to the token it’s inside, and vice versa.

Let’s take a look at an example:

In [ ]:
from transformers import AutoTokenizer
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
print(f"is bert tokenizer fast: {bert_tokenizer.is_fast}")
roberta_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
print(f"is roberta tokenizer fast: {roberta_tokenizer.is_fast}")

## Word <--> Token <--> Sentence

In [ ]:
example = "My name is Sylvain and I work at Hugging Face in Brooklyn."
encoding = bert_tokenizer(example)
print(type(encoding))
print("====================")
print(f"is fast: {encoding.is_fast}")
print("====================")
print(f"tokens: {encoding.tokens()}")
print("====================")
print(f"word ids: {encoding.word_ids()}")
print("====================")
print(f"sentence ids: {encoding.token_type_ids}")

- The tokenizer’s special tokens `[CLS]` and `[SEP]` are mapped to `None`
- Each token is mapped to the word it originates from.
    - This is especially useful to determine if a token is at the start of a word 
    - or if two tokens are in the same word.
    - this method works for **any** type of tokenizer as long as it’s a **fast** one.
- We can use the `##` prefix to map tokens to words, but it only works for **BERT-like** tokenizers;

==============
- The notion of what a word is complicated.
- For instance, does “I’ll” (a contraction of “I will”) count as one or two words?
    - It actually depends on the tokenizer and the pre-tokenization operation it applies.
    - Some tokenizers just split on spaces, so they will consider this as one word.
    - Others use punctuation on top of spaces, so will consider it two words.
- We can see the difference between `bert` and `roberta` tokenizer for `"81s"`

In [ ]:
example = "81s"
encoding = bert_tokenizer(example)
print(f"bert tokens: {encoding.tokens()}")
print(f"bert word ids: {encoding.word_ids()}")
print("====================")
encoding = roberta_tokenizer(example)
print(f"roberta tokens: {encoding.tokens()}")
print(f"roberta word ids: {encoding.word_ids()}")

## Token <--> Char <--> Word

In [ ]:
example = "My name is Sylvain and I work at Hugging Face in Brooklyn."
encoding = bert_tokenizer(example)
print(f"words: {encoding.word_ids()}")
print("=====================")
print(f"tokens: {encoding.tokens()}")
print("=====================")
word_id = 3
s, e = encoding.word_to_chars(word_id)
print(f"word number {word_id}: {example[s:e]}")
print("=====================")
token_id = 5
s,e = encoding.token_to_chars(token_id)
print(f"token number {token_id}: {example[s:e]}")
print("=====================")
char_id = 30
print(f"char {char_id} belongs to word {encoding.char_to_word(char_id)} and token {encoding.char_to_token(char_id)}")

# In the NER pipeline

## Pipeline inference example

The model used by default is [`dbmdz/bert-large-cased-finetuned-conll03-english`](https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english)

In [ ]:
from pprint import pprint
from transformers import pipeline
token_classifier = pipeline("token-classification")
result = token_classifier("My name is Sylvain and I work at Hugging Face in Brooklyn.")
for line in result: print(line)

We can also ask the pipeline to group together the tokens that correspond to the same entity

- The `aggregation_strategy` picked will change the scores computed for each grouped entity.
- With `"simple"` the score is just the mean of the scores of each token in the given entity
    - for instance, the score of “Sylvain” is the mean of the scores we saw in the previous example for the tokens `S`, `##yl`, `##va`, and `##in`.
- Other strategies available are:
    - `"first"`, where the score of each entity is the score of the first token of that entity
    - `"max"`, where the score of each entity is the maximum score of the tokens in that entity
    - `"average"`, where the score of each entity is the average of the scores of the words composing that entity
        - for “Sylvain” there would be no difference from the `"simple"` strategy,
        - but “Hugging Face” would have a score of 0.9819, the average of the scores for “Hugging”, 0.975, and “Face”, 0.98879

In [ ]:
token_classifier = pipeline("token-classification", aggregation_strategy="simple")
agg_result = token_classifier("My name is Sylvain and I work at Hugging Face in Brooklyn.")
for line in agg_result: print(line)

## Inference without pipeline

In [ ]:
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_checkpoint = "dbmdz/bert-large-cased-finetuned-conll03-english"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint)
example = "My name is Sylvain and I work at Hugging Face in Brooklyn."
inputs = tokenizer(
    example,
    return_tensors="pt",
    return_offsets_mapping=True
)


offset_mapping = inputs.pop("offset_mapping")
outputs = model(**inputs)
print("======================")
print(inputs["input_ids"].shape)
print("======================")
probabilities = F.softmax(outputs.logits, dim=-1)[0].data
print(outputs.logits.shape)
print(probabilities.shape)

predictions = outputs.logits.argmax(dim=-1)[0].tolist()
offset_mapping = offset_mapping.squeeze().tolist()

print("======================")
print(model.config.id2label)
results = []
print("======================")
for i, (t, p) in enumerate(zip(inputs.tokens(), predictions)):
    if p != 0:
        results.append(
            {
                "entity": model.config.id2label[p],
                "score": probabilities[i, p],
                "index": i,
                "word": t,
                "start": offset_mapping[i][0],
                "end": offset_mapping[i][1]
            }
        )
for line in results: print(line)

- In the current example we would expect our model to classify the token `S` as `B-PER` (beginning of a person entity) and the tokens `##yl`, `##va` and `##in` as `I-PER` (inside a person entity).
- You might think the model was wrong in this case as it gave the label `I-PER` to all four of these tokens, but that’s not entirely true.
- There are actually two formats for those `B-` and `I-` labels: _IOB1_ and _IOB2_.
- In the _IOB1_ format , the labels beginning with `B-` are only ever used to separate two adjacent entities of the same type.
- The model we are using was fine-tuned on a dataset using that format, which is why it assigns the label `I-PER` to the `S` token.

### Grouping entities

In [ ]:
results = []
prev_class = None

curr_start = None
curr_end = None
score_list = []
print("======================")
for i, (t, p) in enumerate(zip(inputs.tokens(), predictions)):
    label = model.config.id2label[p]
    if label.startswith(("O", "B")): # reset
        if score_list:
            label_list = list(set(label_list))
            if len(label_list) > 1: raise ValueError("multiple labels for an entity")
            results.append(
                {
                    "entity_group": label_list[0],
                    "score": sum(score_list)/len(score_list),
                    "word": example[curr_start: curr_end],
                    "start": curr_start,
                    "end": curr_end,
                }
            )
        curr_start = None
        score_list = []
        label_list = []
    if not label.startswith("O"):
        if not curr_start: curr_start, curr_end = offset_mapping[i]
        else: curr_end = offset_mapping[i][1]
        score_list.append(probabilities[i, p])
        label_list.append(label.split("-")[-1])
for line in results: print(line)

# In the QA Pipeline

## Pipeline inference example

In [ ]:
from transformers import pipeline
default_checkpoint = "distilbert/distilbert-base-cased-distilled-squad"
question_answerer = pipeline("question-answering", model = default_checkpoint)
context = """
🤗 Transformers is backed by the three most popular deep learning libraries — Jax, PyTorch, and TensorFlow — with a seamless integration
between them. It's straightforward to train your models with one before loading them for inference with the other.
"""
question = "Which deep learning libraries back 🤗 Transformers?"
question_answerer(question=question, context=context, top_k=5)

Unlike the other pipelines, which can’t truncate and split texts that are longer than the maximum length accepted by the model (and thus may miss information at the end of a document), this pipeline **can deal with very long contexts** and will return the answer to the question even if it’s at the end:

In [ ]:
with open("qa_sample.txt", "r") as file:
    long_context = file.read()

print(long_context[:200])
print("=======================")
question_answerer(question=question, context=long_context)

## Inference without pipeline

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
tokenizer = AutoTokenizer.from_pretrained(default_checkpoint)
model = AutoModelForQuestionAnswering.from_pretrained(default_checkpoint)
inputs = tokenizer(question, context, return_tensors="pt", return_offsets_mapping=True)
offset_mapping = inputs.pop("offset_mapping")
offset_mapping.squeeze_()
outputs = model(**inputs)

- The model has been trained to predict the index of the token starting the answer (here 21) and the index of the token where the answer ends (here 24).
- This is why those models don’t return one tensor of logits but two: one for the logits corresponding to the start token of the answer, and one for the logits corresponding to the end token of the answer.
- To convert those logits into probabilities, we will apply a softmax function
- Before Softmax, we need to make sure we mask the indices that are not part of the context.
    - Our input is `[CLS] question [SEP] context [SEP]`
    - We need to mask the tokens of the question as well as the `[SEP]` token.
    - We’ll keep the `[CLS]` token, however, as some models use it to indicate that the answer is not in the context.

In [ ]:
import torch
import numpy as np

start_logits = outputs.start_logits
end_logits = outputs.end_logits
print("======================")
print(start_logits.shape, end_logits.shape)

sequence_ids = inputs.sequence_ids()
print("======================")
print(sequence_ids)
# Mask everything apart from the tokens of the context
mask = [i != 1 for i in sequence_ids]
# Unmask the [CLS] token
mask[0] = False
mask = torch.tensor(mask)[None]
print("======================")
print(mask)
start_logits[mask] = -10000
end_logits[mask] = -10000

start_probabilities = torch.nn.functional.softmax(start_logits, dim=-1)[0]
end_probabilities = torch.nn.functional.softmax(end_logits, dim=-1)[0]
print(start_probabilities.shape, end_probabilities.shape)

- At this stage, we could take the argmax of the start and end probabilities
    - But we might end up with a start index that is greater than the end index, so we need to take a few more precautions.
- We will compute the probabilities of each possible `start_index` and `end_index` where `start_index <= end_index`, then take the tuple `(start_index, end_index)` with the highest probability.

- Assuming the events “The answer starts at `start_index`” and “The answer ends at `end_index`” to be independent, the probability that the answer starts at `start_index` and ends at `end_index` is: $start\_probabilities[start\_index]×end\_probabilities[end\_index]$
- Then we set record where `start_index` > `end_index` to zero

In [ ]:
scores = start_probabilities[:, None] * end_probabilities[None, :]
scores = torch.triu(scores)
# most likely k (start, end) ======
def topk_indices(input_tensor, k):
    values, flat_indices = torch.topk(input_tensor.flatten(), k)
    shape = input_tensor.shape
    multi_dim_indices = [np.unravel_index(i.item(), shape) for i in flat_indices]
    return values.data, multi_dim_indices
scores, intervals = topk_indices(scores, 5)

results = []
for score, interval in zip(scores, intervals):
    start_char, _ = offset_mapping[interval[0]]
    _, end_char = offset_mapping[interval[1]]
    answer = context[start_char:end_char]

    result = {
        "answer": answer,
        "start": start_char,
        "end": end_char,
        "score": score,
    }
    results.append(result)
    
for result in results: print(result)

### Handling long contexts

- Assume we impose a max length (#tokens) of 384 for this pipeline (although the current model max legnth is 512)
- We’ll get a number of tokens higher than the imposed maximum length for the long context
- So, we’ll need to truncate our inputs at that maximum length.
- we don’t want to truncate the question, only the context.
    - Since the context is the second sentence, we’ll use the `"only_second"` truncation strategy.
- The problem that arises then is that the answer to the question may not be in the truncated context
    - Which is exactly the case with our question!!!

In [ ]:
inputs = tokenizer(question, long_context)
max_seq_len = 384
print(f"long context length: {len(inputs["input_ids"])}")
print(f"model_max_length: {tokenizer.model_max_length}")
print(f"imposed max length: {max_seq_len}")

inputs = tokenizer(question, long_context, truncation="only_second")
print(tokenizer.decode(inputs["input_ids"]))

- To fix this, the `question-answering` pipeline allows us to split the context into **smaller chunks**, specifying the maximum length.
- To make sure we don’t split the context at exactly the wrong place to make it possible to find the answer, it also includes some **overlap** between the chunks.
    - we would need to add padding to have the last entry be the same size as the others
- We can have the tokenizer (**fast or slow**) do this for us by adding `return_overflowing_tokens=True`
- We can specify the overlap we want with the `stride` argument.

In [ ]:
sentences = [
    "This sentence is not too long but we are going to split it anyway.",
    "This sentence is shorter but will still get split.",
]
inputs = tokenizer(
    sentences,
    truncation=True,
    return_overflowing_tokens=True,
    max_length=6,
    stride=2,
    padding=True
)

print("============================")
for ids in inputs["input_ids"]:
    print(tokenizer.decode(ids))

print("============================")
for k,v in inputs.items(): print(k)

print("============================")
print(inputs["overflow_to_sample_mapping"])

Now let’s go back to our long context.
- By default the `question-answering` pipeline uses a maximum length of 384, and a stride of 128
    - Which correspond to the way the model was fine-tuned
    - You can adjust those parameters by passing `max_seq_len` and `stride` arguments when calling the pipeline.
- We’ll also add padding (to have samples of the same length, so we can build tensors)
- as well as ask for the offsets.

In [ ]:
inputs = tokenizer(
    question,
    long_context,
    stride=128,
    max_length=384,
    padding="longest",
    truncation="only_second",
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
    return_tensors="pt"
)

print("============================")
for k,v in inputs.items(): print(f"{k}: shape = {v.shape}")
print("============================")
print(tokenizer.decode(inputs['input_ids'][0]))
print(tokenizer.decode(inputs['input_ids'][1]))

overflow_to_sample_mapping = inputs.pop("overflow_to_sample_mapping")
offsets = inputs.pop("offset_mapping")

outputs = model(**inputs)
start_logits = outputs.start_logits
end_logits = outputs.end_logits
print("============================")
print(f"start_logits shape: {start_logits.shape}, end_logits shape: {end_logits.shape}")


sequence_ids = inputs.sequence_ids()
# Mask everything apart from the tokens of the context
mask = [i != 1 for i in sequence_ids]
# Unmask the [CLS] token
mask[0] = False
# Mask all the [PAD] tokens
mask = torch.logical_or(torch.tensor(mask)[None], (inputs["attention_mask"] == 0))
print("============================")
print(f"mask shape: {mask.shape}")
start_logits[mask] = -10000
end_logits[mask] = -10000


start_probabilities = torch.nn.functional.softmax(start_logits, dim=-1)
end_probabilities = torch.nn.functional.softmax(end_logits, dim=-1)
print("============================")
print(f"start_probabilities shape: {start_probabilities.shape}")
print(f"end_probabilities shape: {end_probabilities.shape}")

- The next step is similar to what we did for the small context, but we repeat it for each of our two chunks.
- We attribute a score to all possible spans of answer, then take the span with the best score:

#### One candidate per chunk

In [ ]:
candidates = []
for start_probs, end_probs in zip(start_probabilities, end_probabilities):
    scores = start_probs[:, None] * end_probs[None, :]
    idx = torch.triu(scores).argmax().item()

    start_idx = idx // scores.shape[1]
    end_idx = idx % scores.shape[1]
    score = scores[start_idx, end_idx].item()
    candidates.append((start_idx, end_idx, score))
print("=========================")
print(candidates)

print("=========================")
for candidate, offset in zip(candidates, offsets):
    start_token, end_token, score = candidate
    start_char, _ = offset[start_token]
    _, end_char = offset[end_token]
    answer = long_context[start_char:end_char]
    result = {"answer": answer, "start": start_char, "end": end_char, "score": score}
    print(result)

#### Top-K candidates in the entire context

In [ ]:
topk = 5
scores = torch.triu(start_probabilities[..., None] * end_probabilities[:, None])
scores, intervals = topk_indices(scores, topk)

results = []
for score, interval in zip(scores, intervals):
    chunk_id, start_token, end_token = interval
    start_char, _ = offsets[chunk_id][start_token]
    _, end_char = offsets[chunk_id][end_token]
    answer = long_context[start_char:end_char]
    result = {
        "score": score,
        "start": start_char,
        "end": end_char,
        "answer": answer,
    }
    print(result)

##### Compare with the pipeline result

In [ ]:
pipeline_res = question_answerer(question=question, context=long_context, max_seq_len = 384, doc_stride = 128, top_k=5)
for line in pipeline_res: print(line)